In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.feature_selection import mutual_info_regression

# Set style for better visualizations
plt.style.use('bmh')  # Using a built-in style
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['axes.grid'] = True

# Load datasets
train_df = pd.read_csv('/home/USER/Projects/ikaggle/KISS/data/train.csv')
validate_df = pd.read_csv('/home/USER/Projects/ikaggle/KISS/data/validate.csv')

# Merge train and validate datasets
combined_df = pd.concat([train_df, validate_df], ignore_index=True)

# 1. Target Variable Analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Original distribution
ax1.hist(train_df['PremiumAmount'], bins=50, alpha=0.7, color='skyblue')
ax1.set_title('Premium Amount Distribution')
ax1.set_xlabel('Premium Amount')
ax1.set_ylabel('Frequency')

# Log-transformed distribution
log_premium = np.log1p(train_df['PremiumAmount'])
ax2.hist(log_premium, bins=50, alpha=0.7, color='lightgreen')
ax2.set_title('Log-Transformed Premium Distribution')
ax2.set_xlabel('Log Premium Amount')
ax2.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# 2. Correlation Analysis
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns
correlation_matrix = train_df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
im = plt.imshow(correlation_matrix, cmap='RdBu', aspect='auto')
plt.colorbar(im)
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=45)
plt.yticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# 3. Feature Importance Analysis
def calculate_mi_scores(X, y):
    mi_scores = mutual_info_regression(X, y)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    return mi_scores.sort_values(ascending=False)

X = train_df.select_dtypes(include=['float64', 'int64']).drop('PremiumAmount', axis=1)
y = train_df['PremiumAmount']
mi_scores = calculate_mi_scores(X, y)

plt.figure(figsize=(12, 6))
mi_scores.plot(kind='bar', color='darkblue', alpha=0.7)
plt.title('Feature Importance (Mutual Information)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 4. Top Feature Relationships
top_features = mi_scores.head(3).index.tolist()

# Create scatter matrix for top features
fig, axes = plt.subplots(len(top_features), len(top_features), figsize=(15, 15))
fig.suptitle('Feature Relationships Matrix', y=1.02)

for i, feature1 in enumerate(top_features):
    for j, feature2 in enumerate(top_features):
        ax = axes[i, j]
        if i != j:
            ax.scatter(train_df[feature1], train_df[feature2], 
                      c=train_df['PremiumAmount'], 
                      cmap='viridis', alpha=0.5, s=20)
            ax.set_title(f'{feature1} vs {feature2}')
        else:
            ax.hist(train_df[feature1], bins=30)
        if i == len(top_features)-1:
            ax.set_xlabel(feature2)
        if j == 0:
            ax.set_ylabel(feature1)

plt.tight_layout()
plt.show()

# Print statistical summary
print("\nFeature Statistics:")
print(train_df[numeric_cols].describe())

# Print top correlations with target
correlations = train_df[numeric_cols].corr()['PremiumAmount'].sort_values(ascending=False)
print("\nTop Correlations with Premium Amount:")
print(correlations.head())